In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
base_folder = "/content/drive/MyDrive/aiassignment3"
print(os.listdir(base_folder))

['CrimeReport (1).txt', '911-calls-wav2vec2.ipynb', '911_calls_analysis.csv', 'LESO2.pdf', 'student2_mixed_pdf_analysis.csv', 'student2.ipynb', 'student3_final_output.csv', 'student3.ipynb', 'Walk1.mpg', 'Browse2.mpg', 'Rest_InChair.mpg', 'Rest_FallOnFloor.mpg', 'LeftBox.mpg', 'student4_event_log.csv', 'student4.ipynb', 'student5_output.csv', 'student5.ipynb', 'finalintegration.ipynb']


In [4]:
audio_df = pd.read_csv(f"{base_folder}/911_calls_analysis.csv")
pdf_df   = pd.read_csv(f"{base_folder}/student2_mixed_pdf_analysis.csv")
image_df = pd.read_csv(f"{base_folder}/student3_final_output.csv")
video_df = pd.read_csv(f"{base_folder}/student4_event_log.csv")
text_df  = pd.read_csv(f"{base_folder}/student5_output.csv")

print("Audio shape:", audio_df.shape)
print("PDF shape:", pdf_df.shape)
print("Image shape:", image_df.shape)
print("Video shape:", video_df.shape)
print("Text shape:", text_df.shape)

Audio shape: (3, 6)
PDF shape: (61, 8)
Image shape: (154, 5)
Video shape: (27, 6)
Text shape: (115, 6)


In [5]:
print("Audio columns:", audio_df.columns.tolist())
print("PDF columns:", pdf_df.columns.tolist())
print("Image columns:", image_df.columns.tolist())
print("Video columns:", video_df.columns.tolist())
print("Text columns:", text_df.columns.tolist())

Audio columns: ['Call_ID', 'Transcript', 'Extracted_Event', 'Location', 'Sentiment', 'Urgency_Score']
PDF columns: ['Report_ID', 'Start_Page', 'End_Page', 'Department', 'Doc_Type', 'Date', 'Program', 'Key_Detail']
Image columns: ['Image_ID', 'Scene_Type', 'Objects_Detected', 'Bounding_Boxes', 'Confidence']
Video columns: ['Clip_ID', 'Timestamp', 'Frame_ID', 'Event_Detected', 'Persons_Count', 'Confidence']
Text columns: ['Text_ID', 'Crime_Type', 'Location_Entity', 'Sentiment', 'Topic', 'Severity_Label']


In [6]:
audio_clean = audio_df[["Extracted_Event"]].copy()
audio_clean.columns = ["Audio_Event"]

pdf_clean = pdf_df[["Doc_Type"]].copy()
pdf_clean.columns = ["PDF_Doc_Type"]

image_clean = image_df[["Objects_Detected"]].copy()
image_clean.columns = ["Image_Objects"]

video_clean = video_df[["Event_Detected"]].copy()
video_clean.columns = ["Video_Event"]

text_clean = text_df[["Crime_Type"]].copy()
text_clean.columns = ["Text_Crime_Type"]

In [7]:
max_len = max(
    len(audio_clean),
    len(pdf_clean),
    len(image_clean),
    len(video_clean),
    len(text_clean)
)

print("Max rows:", max_len)

Max rows: 154


In [8]:
audio_clean = audio_clean.reindex(range(max_len))
pdf_clean   = pdf_clean.reindex(range(max_len))
image_clean = image_clean.reindex(range(max_len))
video_clean = video_clean.reindex(range(max_len))
text_clean  = text_clean.reindex(range(max_len))

In [9]:
incident_ids = [f"INC_{i+1:03}" for i in range(max_len)]

In [10]:
final_df = pd.DataFrame({
    "Incident_ID": incident_ids,
    "Audio_Event": audio_clean["Audio_Event"],
    "PDF_Doc_Type": pdf_clean["PDF_Doc_Type"],
    "Image_Objects": image_clean["Image_Objects"],
    "Video_Event": video_clean["Video_Event"],
    "Text_Crime_Type": text_clean["Text_Crime_Type"]
})

final_df.head(20)

,Incident_ID,Audio_Event,PDF_Doc_Type,Image_Objects,Video_Event,Text_Crime_Type
0,INC_001,Shooting / active shooter,Training Document,fire,Person Running,Disturbance
1,INC_002,Shooting / active shooter,MRAP Operations,fire,Person Sitting,Disturbance
2,INC_003,Shooting / active shooter,Not found,NaN,Person Running,Disturbance
3,INC_004,NaN,Lesson Plan,fire,Person Walking,Disturbance
4,INC_005,NaN,Training Document,"fire, smoke",Person Sitting,Murder
5,INC_006,NaN,Training Document,smoke,Person Sitting,Disturbance
6,INC_007,NaN,Training Document,fire,Person Running,Robbery
7,INC_008,NaN,Not found,fire,Person Sitting,Disturbance
8,INC_009,NaN,Training Document,fire,Person Walking,Disturbance
9,INC_010,NaN,Not found,NaN,Person Sitting,Disturbance


In [11]:
final_df = final_df.fillna("Not Available")
final_df.head(20)

,Incident_ID,Audio_Event,PDF_Doc_Type,Image_Objects,Video_Event,Text_Crime_Type
0,INC_001,Shooting / active shooter,Training Document,fire,Person Running,Disturbance
1,INC_002,Shooting / active shooter,MRAP Operations,fire,Person Sitting,Disturbance
2,INC_003,Shooting / active shooter,Not found,Not Available,Person Running,Disturbance
3,INC_004,Not Available,Lesson Plan,fire,Person Walking,Disturbance
4,INC_005,Not Available,Training Document,"fire, smoke",Person Sitting,Murder
5,INC_006,Not Available,Training Document,smoke,Person Sitting,Disturbance
6,INC_007,Not Available,Training Document,fire,Person Running,Robbery
7,INC_008,Not Available,Not found,fire,Person Sitting,Disturbance
8,INC_009,Not Available,Training Document,fire,Person Walking,Disturbance
9,INC_010,Not Available,Not found,Not Available,Person Sitting,Disturbance


In [12]:
def compute_severity(row):
    text_val = str(row["Text_Crime_Type"]).lower()
    audio_val = str(row["Audio_Event"]).lower()
    image_val = str(row["Image_Objects"]).lower()
    video_val = str(row["Video_Event"]).lower()
    pdf_val = str(row["PDF_Doc_Type"]).lower()

    high_keywords = [
        "murder", "robbery", "assault", "fire", "shooting",
        "collapse", "collapsing", "fight", "fighting", "trapped"
    ]

    medium_keywords = [
        "theft", "accident", "disturbance", "vehicle", "smoke",
        "burglary", "fraud", "person movement"
    ]

    combined = " ".join([text_val, audio_val, image_val, video_val, pdf_val])

    if any(word in combined for word in high_keywords):
        return "High"
    elif any(word in combined for word in medium_keywords):
        return "Medium"
    else:
        return "Low"

In [13]:
final_df["Severity"] = final_df.apply(compute_severity, axis=1)
final_df.head(20)

,Incident_ID,Audio_Event,PDF_Doc_Type,Image_Objects,Video_Event,Text_Crime_Type,Severity
0,INC_001,Shooting / active shooter,Training Document,fire,Person Running,Disturbance,High
1,INC_002,Shooting / active shooter,MRAP Operations,fire,Person Sitting,Disturbance,High
2,INC_003,Shooting / active shooter,Not found,Not Available,Person Running,Disturbance,High
3,INC_004,Not Available,Lesson Plan,fire,Person Walking,Disturbance,High
4,INC_005,Not Available,Training Document,"fire, smoke",Person Sitting,Murder,High
5,INC_006,Not Available,Training Document,smoke,Person Sitting,Disturbance,Medium
6,INC_007,Not Available,Training Document,fire,Person Running,Robbery,High
7,INC_008,Not Available,Not found,fire,Person Sitting,Disturbance,High
8,INC_009,Not Available,Training Document,fire,Person Walking,Disturbance,High
9,INC_010,Not Available,Not found,Not Available,Person Sitting,Disturbance,Medium


In [14]:
print(final_df.to_string(index=False))

Incident_ID               Audio_Event                 PDF_Doc_Type Image_Objects       Video_Event Text_Crime_Type Severity
    INC_001 Shooting / active shooter            Training Document          fire    Person Running     Disturbance     High
    INC_002 Shooting / active shooter              MRAP Operations          fire    Person Sitting     Disturbance     High
    INC_003 Shooting / active shooter                    Not found Not Available    Person Running     Disturbance     High
    INC_004             Not Available                  Lesson Plan          fire    Person Walking     Disturbance     High
    INC_005             Not Available            Training Document   fire, smoke    Person Sitting          Murder     High
    INC_006             Not Available            Training Document         smoke    Person Sitting     Disturbance   Medium
    INC_007             Not Available            Training Document          fire    Person Running         Robbery     High
    INC_

In [15]:
output_path = f"{base_folder}/final_integrated_incident_dataset.csv"
final_df.to_csv(output_path, index=False)

print("Saved final integrated dataset to:", output_path)

Saved final integrated dataset to: /content/drive/MyDrive/aiassignment3/final_integrated_incident_dataset.csv
